¡Menudo salto de nivel! Acabas de pasar de las finanzas descentralizadas (DeFi) al mundo de **DePIN (Redes de Infraestructura Física Descentralizada)** y los **Oráculos IoT**. Este es exactamente el tipo de arquitectura que empresas reales usan para registrar la huella de carbono o monitorizar cadenas de frío en Ethereum.

Para simular esto con las herramientas que ya tenemos configuradas (Remix, Ganache y Python), vamos a transformar tu bot en un **"Oráculo de Hardware"** (el sensor NDIR) y crearemos un contrato inteligente optimizado para gastar el mínimo Gas posible.

Aquí tienes cómo mapeamos la teoría de tu texto a nuestro simulador:

1. **El Sensor NDIR:** Será tu script de Python generando lecturas de CO2 ficticias.
2. **Cómputo Off-chain:** Python calculará el promedio de CO2 localmente, en lugar de enviar 100 lecturas al contrato para que Solidity haga la matemática (lo cual costaría una fortuna en Gas).
3. **Optimización por Hash y Flags:** Python agrupará las lecturas, creará un sello criptográfico (Hash) y solo actualizará el contrato si el CO2 cambia drásticamente (Patrón Publicar-Suscribir).

A continuación, te presento el código para simular este ecosistema:

### 1. El Contrato Optimizado (`SensorNDIR.sol`)

Crea este archivo en Remix. Fíjate que no guardamos listas gigantes de datos, solo el último estado válido y un Hash para demostrar que no hicimos trampa.

```solidity
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract RegistroCO2 {
    address public oraculoSensor;
    
    // Variables optimizadas (Flags y datos mínimos)
    uint256 public ultimoPromedioCO2;
    uint256 public ultimaActualizacion;
    bytes32 public hashLoteDatos; // El "sello" de los datos crudos
    bool public alertaActiva;     // Flag simple para los suscriptores

    // PATRÓN PUBLISH-SUBSCRIBE: Emitimos un evento que los bots pueden "escuchar" gratis
    event DatosActualizados(uint256 nuevoPromedio, uint256 timestamp, bytes32 hashLote);

    constructor() {
        oraculoSensor = msg.sender; // Solo tú (el sensor físico) puedes escribir aquí
    }

    // Función que el sensor llama solo cuando hay cambios importantes
    function publicarDatos(uint256 _promedio, bytes32 _hashLote) public {
        require(msg.sender == oraculoSensor, "No autorizado. Solo el hardware NDIR puede escribir.");
        
        ultimoPromedioCO2 = _promedio;
        hashLoteDatos = _hashLote;
        ultimaActualizacion = block.timestamp;
        
        // Flag: Si el CO2 sube de 1000 ppm, activamos una alerta (ej. mala ventilación)
        alertaActiva = (_promedio > 1000);

        // Publicamos el evento para los suscriptores
        emit DatosActualizados(_promedio, block.timestamp, _hashLote);
    }
}

```

### 2. El Oráculo IoT / Bot Cómputo Off-chain (`sensor_ndir.py`)

Este script de Python reemplaza a tu bot anterior. Generará datos, los procesará "fuera de la cadena" (Off-chain) y decidirá de forma inteligente si vale la pena gastar Gas para publicarlos.

In [ ]:
from web3 import Web3
import time
import random
import hashlib

# 1. Conexión a Ganache
w3 = Web3(Web3.HTTPProvider('http://127.0.0.1:8545'))

# ⚠️ RELLENA CON TUS DATOS DE GANACHE (Como hicimos antes)
DIRECCION_CONTRATO = Web3.to_checksum_address("0xPegaAquiLaDireccionDelContrato")
CLAVE_PRIVADA_SENSOR = "TuClavePrivadaAqui"
DIRECCION_SENSOR = Web3.to_checksum_address("0xPegaAquiLaCuenta1")

# ABI simplificado del nuevo contrato
abi_sensor = [
    {
        "inputs": [{"internalType": "uint256","name": "_promedio","type": "uint256"},
                   {"internalType": "bytes32","name": "_hashLote","type": "bytes32"}],
        "name": "publicarDatos",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function"
    },
    {
        "inputs": [], "name": "ultimoPromedioCO2",
        "outputs": [{"internalType": "uint256", "name": "", "type": "uint256"}],
        "stateMutability": "view", "type": "function"
    }
]

contrato = w3.eth.contract(address=DIRECCION_CONTRATO, abi=abi_sensor)

print("📡 Sensor NDIR (Oráculo) encendido y conectado.")

# Variable para el patrón Publish-Subscribe (solo publicar si hay cambio > 50 ppm)
UMBRAL_DE_CAMBIO = 50

while True:
    print("\n--- Recopilando datos físicos ---")

    # 1. Simulación de Hardware: Leer CO2 cada segundo durante 5 segundos
    lecturas_crudas = [random.randint(380, 450) for _ in range(5)]
    print(f"Lecturas crudas (ppm): {lecturas_crudas}")

    # 2. CÓMPUTO OFF-CHAIN: Hacemos la matemática en Python, no en Ethereum
    promedio_actual = int(sum(lecturas_crudas) / len(lecturas_crudas))

    # 3. OPTIMIZACIÓN DE GAS: Generar un Hash de los datos crudos
    datos_string = str(lecturas_crudas).encode('utf-8')
    hash_lote = Web3.keccak(text=str(lecturas_crudas)) # Hash keccak256 estándar de Ethereum

    print(f"🧮 Promedio calculado Off-chain: {promedio_actual} ppm")
    print(f"🔒 Hash de integridad: {w3.to_hex(hash_lote)}")

    # 4. PATRÓN PUBLISH-SUBSCRIBE: Decidir si enviamos la transacción
    ultimo_promedio_onchain = contrato.functions.ultimoPromedioCO2().call()

    diferencia = abs(promedio_actual - ultimo_promedio_onchain)

    if diferencia >= UMBRAL_DE_CAMBIO or ultimo_promedio_onchain == 0:
        print(f"⚠️ Cambio significativo detectado ({diferencia} ppm). Actualizando Blockchain...")

        # Enviar transacción (pagar Gas)
        nonce = w3.eth.get_transaction_count(DIRECCION_SENSOR)
        tx = contrato.functions.publicarDatos(promedio_actual, hash_lote).build_transaction({
            'chainId': w3.eth.chain_id,
            'gas': 300000,
            'gasPrice': w3.eth.gas_price,
            'nonce': nonce,
        })

        tx_firmada = w3.eth.account.sign_transaction(tx, CLAVE_PRIVADA_SENSOR)
        tx_hash = w3.eth.send_raw_transaction(tx_firmada.raw_transaction)

        print(f"✅ Datos publicados en la red. Hash TX: {w3.to_hex(tx_hash)}")
    else:
        # Consultas Locales (Polling gratuito)
        print(f"💤 Sin cambios significativos (Dif: {diferencia} ppm). Ahorrando Gas. No se envía transacción.")

    # El sensor físico duerme 10 segundos antes del siguiente escaneo
    time.sleep(10)

### ¿Qué demuestra esta simulación?

Si dejas este bot corriendo, verás en tu terminal cómo a veces lee los datos, calcula el promedio y **decide no hacer nada** ("Ahorrando Gas"). Solo cuando el número aleatorio salta mucho, gasta ETH (falso de Ganache) para firmar la transacción y actualizar el estado global. Esto es aplicar los 4 puntos de tu texto a la perfección.